In [55]:
#Imports
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [1]:
#Initialization of varibales

topic = {
    "array": 0,
    "dictionary" : 0,
    "linked list": 0,
    "tree": 0,
    "graph": 0
}


user_profile = {
    "topic_strength": topic,
    "learning_style": None
}



In [2]:
#main - chatbased CLI
print("Welcome! Let's personalize your CS study notes!Please answer the following questions.")
inintial_topic_questions =["How comfortable are you with Arrays?",
                     "How comfortable are you with Dictionary?",
                     "How comfortable are you with Linked List?",
                     "How comfortable are you with Tree?",
                     "How comfortable are you with Graph?"]
sample_topic_answers = []

initial_personalization_questions = [
    "Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion",
    "Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation"
]

initial_personalization_answers = []
#getting user's input for the initial topic based questions

for i in range(len(inintial_topic_questions)):
    print(inintial_topic_questions[i])
    sample_topic_answers.append(input("Give answer between 1-5"))

for j in range(len(initial_personalization_questions)):
    print(initial_personalization_questions[j])
    initial_personalization_answers.append(input())
    

Welcome! Let's personalize your CS study notes!Please answer the following questions.
How comfortable are you with Arrays?
How comfortable are you with Dictionary?
How comfortable are you with Linked List?
How comfortable are you with Tree?
How comfortable are you with Graph?
Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion
Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation


In [3]:
print(initial_personalization_answers)

['2', '2']


In [4]:
#assign initial topic answers to topic dictionary

for key, value in zip(topic, sample_topic_answers):
    topic[key] = int(value)

print(topic)

for k in range(len(initial_personalization_answers)-1):
    if initial_personalization_answers[k] == "1" and initial_personalization_answers[k+1] == "1":
        user_profile["learning_style"] = "action-based"
    elif initial_personalization_answers[k] == "2" and initial_personalization_answers[k+1] == "2":
        user_profile["learning_style"] ="relationship-based"
    else:
        user_profile["learning_style"] = "mixed"

print(user_profile)

{'array': 1, 'dictionary': 2, 'linked list': 3, 'tree': 4, 'graph': 5}
{'topic_strength': {'array': 1, 'dictionary': 2, 'linked list': 3, 'tree': 4, 'graph': 5}, 'learning_style': 'relationship-based'}


In [5]:
#Generate notes based on weak topics

def find_weakest_topic(user_profile):
    min_key = min(user_profile["topic_strength"], key =user_profile["topic_strength"].get)

    return min_key
print(find_weakest_topic(user_profile))
    

array


In [6]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Access the API key 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [20]:
#Generates notes based on user's learning style and weak topic
full_prompt= f""""
You are an expert Python-based study note generator for computer science students.
Generate concise and helpful study notes on the user's weakest topic: {find_weakest_topic(user_profile)}.
Adapt the tone and structure based on the user's learning style: {user_profile["learning_style"]}.

Use the following formatting rules based on the learning style:

If learning_style is "action-based", structure the output as:

- Simple, direct sentences
- Concise bullet points
- Task- and outcome-focused content
- Give structure coding example

If learning_style is "relationship-based", structure the output as:

- Simple, direct sentences
- A narrative or dialogue-style explanation
- Paragraph format with emotional/contextual cues to build understanding
- Give structure coding example

Ensure the content remains clear, engaging, and easy to follow regardless of the style.
"""


response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

    # Print the response text
response_text = response.choices[0].message.content
print(response_text)

**Study Notes on Arrays for Relationship-Based Learners**

Imagine you’re at a party, and each guest represents an element in an array. Just like how you might arrange guests based on certain criteria—maybe by their height or their age—arrays allow us to organize data in a specific sequence. This organization helps us keep track of information and access it easily.

An array is simply a collection of items stored at contiguous memory locations. Essentially, it allows you to group related data together. For instance, think of your favorite playlist where each song is stored in a certain order. If you want to access a specific song, you just need to know what position it's in. In programming, we do this using indices, starting from zero for the first item.

Let's explore arrays through a practical lens. Imagine you're creating a simple program to store your favorite fruits. You can create an array that holds the names of these fruits:

```python
# Creating an array of fruits
fruits = ["a

In [8]:
#Note feedback loop

survey =[
    {
        "id": 1,
        "question": "How helpful was this note?",
        "options": ["Very helpful and clear", "Somewhat helpful and could be clearer", "Not helpful"]
    },
    {
        "id": 2,
        "question": "How would you describe this note?",
        "options": ["Straight forward and to the point", "Detailed and storylike"]
    },
    {
        "id": 3,
        "question": "What would you prefer more in this note?",
        "options": ["More step by step instructions", "More background, context, stories"]
    },
    {
        "id": 4,
        "question": "Overall does this note match your learning style?",
        "options": ["Perfect match", "Needs more details, stories, example", "Needs more concise example"]
    }
]

In [9]:
#Note survey function
response = []
for i in range(len(survey)):
    print(survey[i]["question"])
    for j in range (len(survey[i]["options"])):
        print(f"{j+1}. {survey[i]["options"][j]}")
    response.append(input("Press only 1 desired number"))


How helpful was this note?
1. Very helpful and clear
2. Somewhat helpful and could be clearer
3. Not helpful
How would you describe this note?
1. Straight forward and to the point
2. Detailed and storylike
What would you prefer more in this note?
1. More step by step instructions
2. More background, context, stories
Overall does this note match your learning style?
1. Perfect match
2. Needs more details, stories, example
3. Needs more concise example


In [21]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)
from llama_index.embeddings.openai import OpenAIEmbedding

In [22]:
embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=5, breakpoint_percentile_threshold=30, embed_model=embed_model
)


In [23]:
from llama_index.core import Document
doc = Document(text=response_text)

In [24]:
nodes = splitter.get_nodes_from_documents([doc]) 

In [40]:
print(nodes[6].get_content())

You can add new guests, just like adding elements to your array using the `append` method. 


In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=500,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

In [46]:
texts = text_splitter.create_documents([response_text])

In [53]:
print(texts[3])
len(texts)

page_content='# Adding a fruit to the list
fruits.append("elderberry")
print(fruits)  # Output: ['apple', 'banana', 'cherry', 'date', 'elderberry']
```

In the code above, each fruit is like a guest at the party, and you can find them by knowing their order. You can add new guests, just like adding elements to your array using the `append` method. This method is a wonderful way to grow your array, just as you might invite new friends to your gathering.'


5

In [52]:
#add faiss to the chunks

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [56]:
#get whats the format for texts

library = FAISS.from_documents(texts, embeddings)

In [57]:
#save faiss index

library.save_local("faiss_index_relationship_chunks")

In [60]:
r_chunk_saved = FAISS.load_local("faiss_index_relationship_chunks", embeddings, allow_dangerous_deserialization=True)

ID: 8d44df0a-1df6-4a39-831c-428bc488d48b
Content: **Study Notes on Arrays for Relationship-Based Learners**

Imagine you’re at a party, and each guest represents an element in an array. Just like how you might arrange guests based on certain criteria—maybe by their height or their age—arrays allow us to organize data in a specific sequence. This organization helps us keep track of information and access it easily.
Metadata: {}
----------------------------------------
ID: 18304c34-fc8c-4713-82f6-02397651752b
Content: An array is simply a collection of items stored at contiguous memory locations. Essentially, it allows you to group related data together. For instance, think of your favorite playlist where each song is stored in a certain order. If you want to access a specific song, you just need to know what position it's in. In programming, we do this using indices, starting from zero for the first item.
Metadata: {}
----------------------------------------
ID: fa8d7b9f-6b0c-44a7-a152-

In [ ]:
for faiss_id in range(r_chunk_saved.index.ntotal):
    # Get the docstore ID
    docstore_id = r_chunk_saved.index_to_docstore_id[faiss_id]
    doc = r_chunk_saved.docstore._dict[docstore_id]
    chunk_text = doc.page_content

    # Prompt for quiz generation
    prompt = f"""
    Based on the following text chunk, generate 2-5 multiple choice questions
    with answers and explanations. Minimum 2 questions, maximum 5 based on relavance. 
    
    - Do not focus on the analogy. Make questions based on programming concept present in the chunk.
    - If you're asking coding based question, make sure to give the relavant code before asking. For example, if you ask what is the output of fruits[0] then surely give fruits array. 
    - Don't make the answer choices obvious. Focus on programming concepts.  

    Text chunk:
    {chunk_text}
    """

    # Call OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a Computer Science quiz generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    # Print the result
    print(f"\n=== Quiz for Chunk {faiss_id} ===")
    print(response.choices[0].message.content)

NameError: name 'r_chunk_saved' is not defined